# 2D Surface Water Flow component


## Overview

This notebook demonstrate the usage of the `river flow dynamics` Landlab component. The component runs a semi-implicit, semi-Lagrangian finite-volume approximation to the depth-averaged 2D shallow-water equations of Casulli and Cheng (1992) and related work.

### Theory

The depth-averaged 2D shallow-water equations are the simplification of the Navier-Stokes equations, which correspond to the balance of momentum and mass in the fluid. It is possible to simplify these equations by assuming a well-mixed water column and a small water depth to width ratio, where a vertical integration results in depth-averaged equations. These require boundary conditions at the top and bottom of the water column, which are provided by the wind stress and the Manning-Chezy formula, respectively:

$$
\frac{\partial U}{\partial t}
+ U\frac{\partial U}{\partial x} + V\frac{\partial U}{\partial y}
= 
- g\frac{\partial \eta}{\partial x}
+ \epsilon\left(\frac{\partial^2 U}{\partial x^2} + \frac{\partial^2 U}{\partial y^2}\right)
+ \frac{\gamma_T(U_a - U)}{H} - g\frac{\sqrt{U^2 + V^2}}{Cz^2}U + \mathbf{f}V
$$

$$
\frac{\partial V}{\partial t}
+ U\frac{\partial V}{\partial x} + V\frac{\partial V}{\partial y}
= 
- g\frac{\partial \eta}{\partial y}
+ \epsilon\left(\frac{\partial^2 V}{\partial x^2} + \frac{\partial^2 V}{\partial y^2}\right)
+ \frac{\gamma_T(V_a - V)}{H} - g\frac{\sqrt{U^2 + V^2}}{Cz^2}V + \mathbf{f}U
$$

$$
\frac{\partial \eta}{\partial t}
+ \frac{\partial (HU)}{\partial x} + \frac{\partial (HV)}{\partial y}
= 0
$$

where $U$ is the water velocity in the $x$-direction, $V$ is the water velocity in the $y$-direction, $H$ is the water depth, $\eta$ is the water surface elevation, $Cz$ is the Chezy friction coefficient, and $t$ is time. For the constants $g$ is the gravity acceleration, $\epsilon$ is the horizontal eddy viscosity, $\mathbf{f}$ is the Coriolis parameter, $\gamma_T$ is the wind stress coefficient, and $U_a$ and $V_a$ are the prescribed wind velocities.

### Numerical representation

A semi-implicit, semi-Lagrangian, finite volume numerical approximation represents the depth averaged, 2D shallow-water equations described before. The water surface elevation, $\eta$, is defined at the center of each computational volume (nodes). Water depth, $H$, and velocity components, $U$ and $V$, are defined at the midpoint of volume faces (links). The finite volume structure provides a control volume representation that is inherently mass conservative.

The combination of a semi-implciit water surface elevation solution and a semi-Lagrangian representation of advection provides the advantages of a stable solution and of time steps that exceed the CFL criterion. In the semi-implicit process, $\eta$ in the momentum equations, and the velocity divergence in the continuity equation, are treated implicitly. The advective terms in the momentum equations, are discretized explicitly. See the cited literature for more details.

### The component

Import the needed libraries:

In [ ]:
import time

import matplotlib.pyplot as plt
import numpy as np
from IPython.display import clear_output
from mpl_toolkits.axes_grid1 import make_axes_locatable

from landlab import RasterModelGrid
from landlab.components import RiverFlowDynamics
from landlab.plot.imshow import imshow_grid
from landlab.io import esri_ascii

## Information about the component

Using the class name as argument for the `help` function returns descriptions of the various methods and parameters.

In [ ]:
help(RiverFlowDynamics)

## Examples

### Example 1: Flow in a rectangular channel 6.0 m long 

This first basic example illustrates water flowing through a rectangular channel 1.0 m wide and 6.0 m long. The channel is made of smooth concrete (Manning's n = 0.012 s/m^(1/3)) with a slope of 0.01 m/m.

We specify basic parameters such as the grid resolution, number of time steps, and domain dimensions.

In [ ]:
# Basic parameters
mannings_n = 0.012  # Manning's roughness coefficient, [s/m^(1/3)]
channel_slope = 0.01  # Channel slope [m/m]

# Simulation parameters
n_timesteps = 1000  # Number of timesteps
dt = 0.1  # Timestep duration, [s]
display_dt = 1.0  # Re-draw the plot every this many simulated seconds
nrows = 20  # Number of node rows
ncols = 60  # Number of node cols
dx = 0.1  # Node spacing in the x-direction, [m]
dy = 0.1  # Node spacing in the y-direction, [m]

Create the grid:

In [ ]:
# Create and set up the grid
grid = RasterModelGrid((nrows, ncols), xy_spacing=(dx, dy))

Create the elevation field and define the topography to represent our rectangular channel:

In [ ]:
# 6 m long channel (60 cols × 0.1 m). Longer domain allows the
# development of the water profile
te = grid.add_field(
    "topographic__elevation", 1.0 - channel_slope * grid.x_of_node, at="node"
)
te[grid.y_of_node > 1.5] = 2.5
te[grid.y_of_node < 0.5] = 2.5

We show a top view of the domain:

In [ ]:
# Determine grid dimensions to find the centerline
nrows, ncols = grid.shape
mid_row = nrows // 2

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4))

# --- LEFT PANEL: Top-down Topography ---
plt.sca(ax1)
imshow_grid(
    grid, "topographic__elevation", cmap="terrain", colorbar_label="Elevation (m)"
)
ax1.set_title("Channel Topography")

# --- RIGHT PANEL: Longitudinal Centerline Profile ---
x_profile = grid.x_of_node.reshape((nrows, ncols))[mid_row, :]
z_profile = grid.at_node["topographic__elevation"].reshape((nrows, ncols))[mid_row, :]
fixed_y_min = z_profile.min() - 0.2

ax2.plot(x_profile, z_profile, color="saddlebrown", lw=2, label="Bed Surface")
ax2.fill_between(x_profile, fixed_y_min, z_profile, color="saddlebrown", alpha=0.4)
ax2.set_title("Longitudinal Profile (Centerline)")
ax2.set_xlabel("Distance (m)")
ax2.set_ylabel("Elevation (m)")
ax2.set_ylim(fixed_y_min, z_profile.max() + 0.5)
ax2.legend()
ax2.grid(True, linestyle="--", alpha=0.5)

plt.tight_layout()
plt.show()

The channel is empty at the beginning of the simulation, so we create the fields for the water surface elevation, depth and velocity:

In [ ]:
# We establish the initial conditions, which represent an empty channel
h = grid.add_zeros("surface_water__depth", at="node")

# Water velocity is zero in everywhere since there is no water yet
vel = grid.add_zeros("surface_water__velocity", at="link")

# Calculating the initial water surface elevation from water depth and topographic elevation
wse = grid.add_field("surface_water__elevation", te, at="node")

Then, we specify the nodes at which water is entering into the domain, and also the associated links. These are going to be the entry boundary conditions for water depth and velocity. In this case, water flows from left to right at 0.5 $m$ depth, with a velocity of 0.45 $m/s$:

In [ ]:
# We set fixed boundary conditions, specifying the nodes and links in which the water is flowing into the grid
fixed_entry_nodes = np.array([300, 360, 420, 480, 540, 600, 660, 720, 780, 840, 900])
fixed_entry_links = grid.links_at_node[fixed_entry_nodes][:, 0]

# We set the fixed values in the entry nodes/links
entry_nodes_h_values = np.array([0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5])
entry_links_vel_values = np.array(
    [0.45, 0.45, 0.45, 0.45, 0.45, 0.45, 0.45, 0.45, 0.45, 0.45, 0.45]
)

And now we show the boundary condition in the cross-section:

In [ ]:
# Extracting the coordinates for cleaner plotting
y_ground = grid.y_of_node[grid.nodes_at_left_edge]
z_ground = te[grid.nodes_at_left_edge]

constant_wse = (entry_nodes_h_values + te[fixed_entry_nodes])[0]

y_intersections = []
for i in range(len(y_ground) - 1):
    if (z_ground[i] - constant_wse) * (z_ground[i + 1] - constant_wse) <= 0:
        slope = (z_ground[i + 1] - z_ground[i]) / (y_ground[i + 1] - y_ground[i])
        exact_y = y_ground[i] + (constant_wse - z_ground[i]) / slope
        y_intersections.append(exact_y)

fig, ax = plt.subplots(figsize=(6.5, 3.9))
ax.fill_between(y_ground, 0.75, z_ground, color="saddlebrown", alpha=0.4, zorder=1)
ax.plot(
    y_ground,
    z_ground,
    color="saddlebrown",
    linewidth=2.5,
    label="Channel Bed",
    zorder=3,
)

if len(y_intersections) >= 2:
    ax.plot(
        [y_intersections[0], y_intersections[-1]],
        [constant_wse, constant_wse],
        color="dodgerblue",
        linewidth=2.5,
        label="Water Surface",
        zorder=4,
    )

    wse_array = np.full_like(z_ground, constant_wse)
    ax.fill_between(
        y_ground,
        z_ground,
        wse_array,
        where=(z_ground <= wse_array),
        color="deepskyblue",
        alpha=0.5,
        interpolate=True,
        zorder=2,
    )

ax.set_title("Channel Cross-section at Entry Boundary", fontsize=13, fontweight="bold")
ax.set_xlabel("Distance across channel [m]", fontsize=11)
ax.set_ylabel("Elevation [m]", fontsize=11)
ax.set_xlim(0.25, 1.75)
ax.set_ylim(0.75, 2.75)
ax.grid(True, linestyle="--", alpha=0.6, zorder=0)
ax.legend(loc="upper center", framealpha=1.0)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
plt.tight_layout()
plt.show()

We construct our component by passing the arguments we defined previously:

In [ ]:
# Finally, we run the model and let the water fill our channel
rfd = RiverFlowDynamics(
    grid,
    dt=dt,
    mannings_n=mannings_n,
    fixed_entry_nodes=fixed_entry_nodes,
    fixed_entry_links=fixed_entry_links,
    entry_nodes_h_values=entry_nodes_h_values,
    entry_links_vel_values=entry_links_vel_values,
)

And finally, we run the simulation for 100 timesteps (10 seconds).


In [ ]:
# Pre-calculate grid dimensions and fixed limits for the profile
nrows, ncols = grid.shape
mid_row = nrows // 2
z_profile_initial = grid.at_node["topographic__elevation"].reshape((nrows, ncols))[
    mid_row, :
]

fixed_x_max = np.max(grid.x_of_node)
fixed_y_min = np.min(z_profile_initial) - 0.2
fixed_y_max = np.max(z_profile_initial) + 1.0


def update_display(elapsed, t0, show_progress=False):
    clear_output(wait=True)
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4))

    plt.sca(ax1)
    grid.imshow("surface_water__depth", limits=(0.0, 0.6))
    ax1.set_title(f"Water Depth (m)  [t = {elapsed:.2f} s / {target_time:.0f} s]")

    x_profile = grid.x_of_node.reshape((nrows, ncols))[mid_row, :]
    z_profile = grid.at_node["topographic__elevation"].reshape((nrows, ncols))[
        mid_row, :
    ]
    wse_profile = grid.at_node["surface_water__elevation"].reshape((nrows, ncols))[
        mid_row, :
    ]
    wse_clean = np.maximum(wse_profile, z_profile)

    h_n_ref = 0.1145  # normal depth [m]: h_n=(n*q/S^0.5)^(3/5), Fr=1.85
    h_c_ref = 0.1728  # critical depth [m]: h_c=(q²/g)^(1/3)
    wse_n = z_profile + h_n_ref
    wse_c = z_profile + h_c_ref

    ax2.plot(x_profile, z_profile, color="saddlebrown", lw=2, label="Bed Surface")
    ax2.fill_between(x_profile, fixed_y_min, z_profile, color="saddlebrown", alpha=0.4)
    ax2.plot(x_profile, wse_clean, color="deepskyblue", lw=2, label="Water Surface")
    ax2.fill_between(x_profile, z_profile, wse_clean, color="deepskyblue", alpha=0.5)
    ax2.plot(
        x_profile,
        wse_n,
        color="limegreen",
        lw=1.5,
        ls="--",
        label="Normal WSE (h_n=0.114 m, Fr=1.85)",
    )
    ax2.plot(
        x_profile,
        wse_c,
        color="orange",
        lw=1.5,
        ls=":",
        label="Critical WSE (h_c=0.173 m)",
    )
    ax2.set_xlim(0, fixed_x_max)
    ax2.set_ylim(fixed_y_min, fixed_y_max)
    ax2.set_title("Longitudinal Profile (Centerline)")
    ax2.set_xlabel("Distance [m]")
    ax2.set_ylabel("Elevation [m]")
    ax2.legend(loc="upper right")

    plt.tight_layout()
    plt.show()

    if show_progress:
        wall = time.time() - t0
        pct = elapsed / target_time
        eta = wall * (1.0 / pct - 1.0) if pct > 0 else float("nan")
        print(
            f"sim-time {elapsed:.2f} / {target_time:.1f} s  "
            f"({pct:.1%})  wall {wall:.1f} s  ETA {eta:.1f} s  "
            f"dt={rfd.current_dt:.4f} s"
        )


# ─── Run loop: advance until target_time is reached ─────────────────────
t0 = time.time()
next_display_t = 0.0  # next simulated-time threshold to redraw
target_time =n_timesteps * dt 
update_display(0.0, t0, show_progress=False)

while rfd.elapsed_time < target_time:
    rfd.run_one_step()

    if rfd.elapsed_time >= next_display_t:
        update_display(rfd.elapsed_time, t0, show_progress=True)
        next_display_t += display_dt

update_display(rfd.elapsed_time, t0, show_progress=False)  # final frame
print("Done.")

Exploring the water depth results at the latest time:

In [ ]:
grid.imshow("surface_water__depth")

And the water surface elevation:

In [ ]:
grid.imshow("surface_water__elevation")

## Example 2: Surface water flowing over a DEM

On this case, we will import a digital elevation model (DEM) for a side-channel of the Kootenai River, Idaho, US.

In [ ]:
# Getting the grid and some parameters
asc_file = "DEM-kootenai_37x50_1x1.asc"
with open(asc_file) as fp:
    grid = esri_ascii.load(fp, name="topographic__elevation")
te = grid.at_node["topographic__elevation"]

Again, we specify some basic parameters such as the time step number and duration. For simplicity, we will keep our previous Manning's coefficient. Notice that we already loaded all the required libraries.

In [ ]:
# Basic parameters
mannings_n = 0.012  # Manning's roughness coefficient, [s/m^(1/3)]

# Simulation parameters
n_timesteps = 75  # Number of timesteps
dt = 1.0  # Timestep duration, [s]
display_animation_freq = 10  # Redraw every this many simulated seconds

Let's see our new topography:

In [ ]:
# Showing the topography
grid.imshow("topographic__elevation")

Our side-channel is empty at the beginning  of the simulation, so we create the proper fields:

In [ ]:
# We establish the initial conditions, which represent an empty channel
h = grid.add_zeros("surface_water__depth", at="node")

# Water velocity is zero in everywhere since there is no water yet
vel = grid.add_zeros("surface_water__velocity", at="link")

# Calculating the initial water surface elevation from water depth and topographic elevation
wse = grid.add_field("surface_water__elevation", te, at="node")

Then, we specify the nodes at which water is entering into the domain, and also the associated links. These are going to be our entry boundary conditions for water depth and velocity. On this case, water flows from right to left:

In [ ]:
# We set fixed boundary conditions, specifying the nodes and links in which the water is flowing into the grid
fixed_entry_nodes = grid.nodes_at_right_edge
fixed_entry_links = grid.links_at_node[fixed_entry_nodes][:, 2]

# We set the fixed values in the entry nodes/links
entry_nodes_h_values = np.array(
    [
        0.0,
        0.0,
        0.0,
        0.0,
        0.0,
        0.0,
        0.0,
        0.0,
        0.0,
        0.0,
        0.0,
        0.0,
        0.04998779,
        0.05999756,
        0.03997803,
        0.0,
        0.0,
        0.0,
        0.05999756,
        0.10998535,
        0.12994385,
        0.09997559,
        0.15997314,
        0.23999023,
        0.30999756,
        0.36999512,
        0.45996094,
        0.50994873,
        0.54998779,
        0.59997559,
        0.63995361,
        0.65997314,
        0.65997314,
        0.60998535,
        0.5,
        0.13995361,
        0.0,
    ]
)
entry_links_vel_values = np.array(
    [
        0.0,
        0.0,
        0.0,
        0.0,
        0.0,
        0.0,
        0.0,
        0.0,
        0.0,
        0.0,
        0.0,
        0.0,
        -2.58638018,
        -2.58638018,
        -2.58638018,
        0.0,
        0.0,
        0.0,
        -2.58638018,
        -2.58638018,
        -2.58638018,
        -2.58638018,
        -2.58638018,
        -2.58638018,
        -2.58638018,
        -2.58638018,
        -2.58638018,
        -2.58638018,
        -2.58638018,
        -2.58638018,
        -2.58638018,
        -2.58638018,
        -2.58638018,
        -2.58638018,
        -2.58638018,
        -2.58638018,
        0.0,
    ]
)

Now we can plot our entry boundary condition in the cross-section:

In [ ]:
plt.plot(
    grid.y_of_node[fixed_entry_nodes], entry_nodes_h_values + te[fixed_entry_nodes]
)
plt.plot(grid.y_of_node[grid.nodes_at_right_edge], te[grid.nodes_at_right_edge])
plt.title("Entry cross-section")
plt.xlabel("Distance [m]")
plt.ylabel("Elevation [m]")
plt.grid(True)

Then we create the component by passing the arguments defined previously:

In [ ]:
# Finally, we run the model and let the water fill our channel
rfd = RiverFlowDynamics(
    grid,
    dt=dt,
    mannings_n=mannings_n,
    fixed_entry_nodes=fixed_entry_nodes,
    fixed_entry_links=fixed_entry_links,
    entry_nodes_h_values=entry_nodes_h_values,
    entry_links_vel_values=entry_links_vel_values,
)

And we run 100 time steps of 0.1 $s$ duration:

In [ ]:
t0 = time.time()
next_display_t = 0.0
nrows, ncols = grid.shape
X = grid.x_of_node.reshape((nrows, ncols))
Y = grid.y_of_node.reshape((nrows, ncols))
Z = grid.at_node["topographic__elevation"].reshape((nrows, ncols))

while rfd.elapsed_time < target_time:
    rfd.run_one_step()

    if rfd.elapsed_time >= next_display_t:
        clear_output(wait=True)

        elapsed_wall = time.time() - t0
        pct = rfd.elapsed_time / target_time
        eta = elapsed_wall * (1.0 / pct - 1.0) if pct > 0 else float("nan")
        print(
            f"sim-time {rfd.elapsed_time:.2f} / {target_time:.1f} s  "
            f"({pct:.1%})  wall {elapsed_wall:.1f} s  ETA {eta:.1f} s  "
            f"dt={rfd.current_dt:.4f} s"
        )

        h_arr = grid.at_node["surface_water__depth"].reshape((nrows, ncols))
        wse = grid.at_node["surface_water__elevation"].reshape((nrows, ncols))
        wse_masked = np.ma.masked_where(h_arr <= 0.001, wse)

        fig, ax = plt.subplots(figsize=(6, 6))
        bed = ax.pcolormesh(X, Y, Z, cmap="gray", shading="auto", vmin=538, vmax=543)
        wat = ax.pcolormesh(
            X,
            Y,
            wse_masked,
            cmap="Blues",
            shading="auto",
            alpha=0.8,
            vmin=538,
            vmax=540,
        )

        divider = make_axes_locatable(ax)
        cax1 = divider.append_axes("right", size="5%", pad=0.15)
        cax2 = divider.append_axes("right", size="5%", pad=0.85)
        fig.colorbar(wat, cax=cax1, label="Water Surface Elevation (m)")
        fig.colorbar(bed, cax=cax2, label="Bed Elevation (m)")

        ax.set_aspect("equal", adjustable="box")
        ax.set_xlabel("X")
        ax.set_ylabel("Y")
        ax.set_title(
            f"Hydrodynamic Routing  [t = {rfd.elapsed_time:.2f} s]", fontsize=11
        )
        plt.tight_layout()
        plt.show()

        next_display_t += display_animation_freq

Finally, we can explore the results by plotting the resulting water depth:

In [ ]:
grid.imshow("surface_water__depth")

### And that's it! 

Nice work completing this tutorial. You know now how to use the `RiverFlowDynamics` Landlab component to run your own simulations :)

-- --



### Click here for more <a href="https://landlab.csdms.io/tutorials/">Landlab tutorials</a>